In [0]:
%restart_python
 

In [0]:

%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bronze;
CREATE SCHEMA IF NOT EXISTS workspace.silver;
CREATE SCHEMA IF NOT EXISTS workspace.gold;

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import initcap, col, lower, upper, trim, concat_ws, levenshtein, when, lit, count, sum as _sum, avg, round, to_date
#from pyspark.sql.types import *  revoir 
#from pyspark.sql.functions import *
import pandas as pd
from datetime import datetime
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, TimestampType, DoubleType
from pyspark.sql import functions as F


# Importer des fichiers des dossiers SRC et TESTS


import sys
import os

#   Récupérer le chemin absolu du dossier courant (NOTEBOOK)
current_dir = os.getcwd()

#  CORRECTION : Remonter d'un niveau (dossier parent) avant d'aller dans SRC
src_path = os.path.abspath(os.path.join(current_dir, "..", "SRC"))


#  L'ajouter au chemin système
if src_path not in sys.path:
    sys.path.append(src_path)

print("----SRC path -------------")
print(src_path)




notebook_path = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .notebookPath()
    .get()
)

project_path = notebook_path.split("/NOTEBOOKS/")[0]

data_dir = (
    f"file:/Workspace{project_path}"
    "/DATA/SAMPLE/ABASSURANCE"
)


#==================================
# Importations des Données oracle
#==================================

# Importer à partir du dossier SRC

from SONAR_FONC_INGESTION_DONNEES_ORACLE import create_ab_dataframes


df_ab_clients, df_ab_contrats, df_ab_sinistres, df_ab_payments = create_ab_dataframes(spark,data_dir)

# Affichage des Dataframes
print("\n # Affichage des Dataframes, 5 premières Lignes ")

print(df_ab_clients.show(5,truncate = False))
print(df_ab_contrats.show(5,truncate=False))
print(df_ab_sinistres.show(5,truncate=False))
print(df_ab_payments.show(5,truncate=False))


# Protection des données
def protection_bronze(df2, source):
    
    salt = dbutils.secrets.get(scope="PROJET_BIG_DATA_STUDI", key="hash_salt")
    
    if source == "Oracle":
    

     df2 = df2.withColumns({"Email_Hached": F.sha2(F.concat(F.col("AB_EMAIL"), F.lit(salt)), 256),

     "Nom_Hached": F.sha2(F.concat(F.col("AB_NOM"), F.lit(salt)), 256),
     "Prenom_Hached": F.sha2(F.concat(F.col("AB_PRENOM"), F.lit(salt)), 256),
     "Adresse_Hached": F.sha2(F.concat(F.col("AB_ADRESSE"), F.lit(salt)), 256),
     "Telephone_Hached": F.sha2(F.concat(F.col("AB_TELEPHONE"), F.lit(salt)), 256),
     "Num_fiscal_Hached": F.sha2(F.concat(F.col("AB_NUM_FISCAL"), F.lit(salt)), 256),
     "Date_de_naissance_Hached": F.sha2(F.concat(F.col("AB_DATE_NAISSANCE"), F.lit(salt)), 256)}).drop("AB_NOM", "AB_PRENOM","AB_EMAIL","AB_DATE_NAISSANCE","AB_ADRESSE","AB_NUM_FISCAL","AB_TELEPHONE")
    else: 
    
     df2 = df2.withColumns({"Email_Hached": F.sha2(F.concat(F.col("AP_MAIL_ADDRESS"), F.lit(salt)), 256),
     "FULL_NAME_Hached": F.sha2(F.concat(F.col("AP_FULL_NAME"), F.lit(salt)), 256),
     "Nom_Hached": F.sha2(F.concat(F.col("AP_NOM"), F.lit(salt)), 256),
     "Prenom_Hached": F.sha2(F.concat(F.col("AP_PRENOM"), F.lit(salt)), 256),
     "Telephone_Hached": F.sha2(F.concat(F.col("AP_PHONE_NUMBER"), F.lit(salt)), 256),
     "Street_Hached": F.sha2(F.concat(F.col("AP_STREET_ADDRESS"), F.lit(salt)), 256),
     "Date_de_naissance_Hached": F.sha2(F.concat(F.col("AP_BIRTH_DATE"), F.lit(salt)), 256)}).drop("AP_FULL_NAME","AP_MAIL_ADDRESS","AP_BIRTH_DATE","AP_NOM", "AP_PRENOM","AP_PHONE_NUMBER","AP_STREET_ADDRESS")
    return df2

# Protection bronze

df_ab_clients_bronze = protection_bronze(df_ab_clients,"Oracle")


# Affichage des DataFrames après le hachage
print("=== df_ab_clients_protege ===")
print(df_ab_clients_bronze.show(5, truncate=False))

print("=== df_ab_contrats ===")
print(df_ab_contrats.show(5, truncate=False))

print("=== df_ab_sinistres ===")
print(df_ab_sinistres.show(5, truncate=False))

print("=== df_ab_payments ===")
print(df_ab_payments.show(5, truncate=False))

df_ab_contrats_bronze = df_ab_contrats
df_ab_sinistres_bronze = df_ab_sinistres
df_ab_payments_bronze= df_ab_payments

#     ENREGISTREMENT EN BRONZE   DES DONNEES ORACLE    
from SONAR_FONC_ENREGISTREMENT_EN_BRONZE import enregistrement_bronze 

# Tables Oracles
enregistrement_bronze(df_ab_clients_bronze,"workspace.bronze.oracle_clients")
enregistrement_bronze(df_ab_contrats_bronze,"workspace.bronze.oracle_contrats")
enregistrement_bronze(df_ab_sinistres_bronze,"workspace.bronze.oracle_sinistres")
enregistrement_bronze(df_ab_payments_bronze,"workspace.bronze.oracle_payments")


#============================================
# Ingestion des données SQLServer
#============================================

# Import load_ap_data 
from SONAR_FONC_INGESTION_DES_DONNEES_ASSUREPLUS import load_ap_data

# Obtention des dataframes
df_ap_clients, df_ap_contrats, df_ap_sinistres, df_ap_payments = load_ap_data(spark,data_dir)

 # Scinder en nom et prénom le full name de Assurplus
def split_full_name(df,col_last_name, col_first_name,col_full_name):

 df = df.withColumn(col_last_name, F.trim(F.split(F.col(col_full_name), " ").getItem(1))) \
             .withColumn(col_first_name, F.trim(F.split(F.col(col_full_name), " ").getItem(0)))
 return df
 # Extraire le nom et prénom à partir du Full_name
df_ap_clients = split_full_name(df_ap_clients,"AP_NOM","AP_PRENOM","AP_FULL_NAME")


# Hachage des données

df_ap_clients_bronze = protection_bronze(df_ap_clients,"SQLServer")



# Affichage df_ap_clients_bronze 
print("=== ASSUREPLUS - Clients ===")
print(df_ap_clients_bronze.show(5, truncate=False))

print("=== ASSUREPLUS - Contrats ===")
print(df_ap_contrats.show(5, truncate=False))

print("=== ASSUREPLUS - Sinistres ===")
print(df_ap_sinistres.show(5, truncate=False))

print("=== ASSUREPLUS - Payments ===")
print(df_ap_payments.show(5, truncate=False))


# ENREGISTREMENT EN BRONZE DES DONNEES SQLServer

df_ap_contrats_bronze = df_ap_contrats
df_ap_sinistres_bronze = df_ap_sinistres
df_ap_payments_bronze= df_ap_payments
      
# Tables SQL Server
enregistrement_bronze(df_ap_clients_bronze,"workspace.bronze.sqlserver_clients")
enregistrement_bronze(df_ap_contrats_bronze,"workspace.bronze.sqlserver_contrats")
enregistrement_bronze(df_ap_payments_bronze,"workspace.bronze.sqlserver_payments")
enregistrement_bronze(df_ap_sinistres_bronze,"workspace.bronze.sqlserver_sinistres")



In [0]:
#========================================================
#   COUCHE SILVER 
#=====================================================



# Ajout du dossier SRC au sys.path pour pouvoir importer les modules du projet
import sys
import os

src_path = os.path.abspath(os.path.join(os.getcwd(), "..", "SRC"))
if src_path not in sys.path:
    sys.path.append(src_path)

#*****************************************************************************
# SUPPRESSION LIGNES INCOHERENTES : les lignes qui présentent des incohérences
# *****************************************************************************

from SONAR_FONC_SUPPRESSION_INCOHERENCES import suppression_incoherences_clients, suppression_incoherences           



# ====================================================
# SUPPRESSION INCOHERENCES CLIENTS AbAssurance
# ====================================================

# Les controles à effectuer pour détecter les lignes incohérentes
controles_ab_clients = [

    # Identifiant client
    (
        F.col("AB_CLIENT_ID").isNull() |
        (F.col("AB_CLIENT_ID") <= 0),
        "Identifiant client invalide"
    ),

    # Nom
    (
        F.col("AB_NOM").isNull() |
        (F.trim(F.col("AB_NOM")) == ""),
        "Nom client manquant"
    ),

    # Prénom
    (
        F.col("AB_PRENOM").isNull() |
        (F.trim(F.col("AB_PRENOM")) == ""),
        "Prénom client manquant"
    ),

    # Email : validation uniquement si renseignée (accents acceptés dans la partie locale : inès.david1@...)
    (
        F.col("AB_EMAIL").isNotNull() &
        (F.trim(F.col("AB_EMAIL")) != "") &
        (~F.trim(F.col("AB_EMAIL")).rlike(
            r"^[A-Za-z0-9._%+\-À-ÖØ-öø-ÿ]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
        )),
        "Adresse email invalide"
    ),

    # Téléphone
    (
        F.col("AB_TELEPHONE").isNotNull() &
        (F.trim(F.col("AB_TELEPHONE")) != "") &
        (~F.regexp_replace(
            F.col("AB_TELEPHONE"), r"[\s.-]", ""
        ).rlike(r"^(0[1-9]\d{8}|\+33[1-9]\d{8}|0033[1-9]\d{8})$")),
        "Numéro de téléphone invalide"
    ),

    # Code postal
    (
        F.col("AB_CODE_POSTAL").isNotNull() &
        (F.trim(F.col("AB_CODE_POSTAL")) != "") &
        (~F.col("AB_CODE_POSTAL").rlike(r"^\d{5}$")),
        "Code postal invalide"
    ),

    # Numéro fiscal
    (
        F.col("AB_NUM_FISCAL").isNotNull() &
        (F.trim(F.col("AB_NUM_FISCAL")) != "") &
        (~F.col("AB_NUM_FISCAL").rlike(r"^\d{13}$")),
        "Numéro fiscal invalide"
    ),

    # Adresse
    (
        F.col("AB_ADRESSE").isNotNull() &
        (F.trim(F.col("AB_ADRESSE")) == ""),
        "Adresse vide"
    ),

    # Statut client
    (
        F.col("AB_STATUT_CLIENT").isNull() |
        (F.trim(F.col("AB_STATUT_CLIENT")) == ""),
        "Statut client manquant"
    )
]


# Exécution du traitement pour  AB Assurance
df_ab_clients_clean, df_ab_clients_Rejected = suppression_incoherences_clients(
    df_ab_clients,
    controles_ab_clients,
    date_naissance="AB_DATE_NAISSANCE",
    date_creation="AB_DATE_CREATION",
    cle_primaire="AB_CLIENT_ID"
)
# Par la suite , on ne trvaillera qu'avec des données valides, les enregistrements Rejected seront sauvegardés pour une anlyse ultérieure
df_ab_clients = df_ab_clients_clean


# ====================================================
# SUPPRESSION INCOHERENCES USERS AssurePlus
# ====================================================

controles_ap_clients = [

    # Identifiant utilisateur
    (
        F.col("AP_USER_ID").isNull() |
        (F.col("AP_USER_ID") <= 0),
        "Identifiant utilisateur invalide"
    ),

    # Nom complet
    (
        F.col("AP_FULL_NAME").isNull() |
        (F.trim(F.col("AP_FULL_NAME")) == ""),
        "Nom complet manquant"
    ),

    # Email
    (
        F.col("AP_MAIL_ADDRESS").isNotNull() &
        (F.trim(F.col("AP_MAIL_ADDRESS")) != "") &
        (~F.trim(F.col("AP_MAIL_ADDRESS")).rlike(
            r"^[A-Za-z0-9._%+\-À-ÖØ-öø-ÿ]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
        )),
        "Adresse email invalide"
    ),

    # Téléphone
    (
        F.col("AP_PHONE_NUMBER").isNotNull() &
        (F.trim(F.col("AP_PHONE_NUMBER")) != "") &
        (~F.regexp_replace(
            F.col("AP_PHONE_NUMBER"), r"[\s.-]", ""
        ).rlike(r"^(0[1-9]\d{8}|\+33[1-9]\d{8}|0033[1-9]\d{8})$")),
        "Numéro de téléphone invalide"
    ),

    # Code postal
    (
        F.col("AP_ZIP_CODE").isNotNull() &
        (F.trim(F.col("AP_ZIP_CODE")) != "") &
        (~F.col("AP_ZIP_CODE").rlike(r"^\d{5}$")),
        "Code postal invalide"
    ),

    # Adresse
    (
        F.col("AP_STREET_ADDRESS").isNotNull() &
        (F.trim(F.col("AP_STREET_ADDRESS")) == ""),
        "Adresse vide"
    ),

    # Statut client
    (
        F.col("AP_CUSTOMER_STATUS").isNull() |
        (F.trim(F.col("AP_CUSTOMER_STATUS")) == ""),
        "Statut client manquant"
    ),

    # Score fidélité : valeur comprise entre 0 et 100 (try_cast : 'inconnu' -> NULL donc invalide)
    (
        F.col("AP_LOYALTY_SCORE").isNull() |
        F.expr("try_cast(AP_LOYALTY_SCORE as int)").isNull() |
        (F.expr("try_cast(AP_LOYALTY_SCORE as int)") < 0) |
        (F.expr("try_cast(AP_LOYALTY_SCORE as int)") > 100),
        "Score fidélité invalide"
    )
]


# Exécution du traitement les clients AssurePlus
df_ap_clients_clean, df_ap_clients_Rejected = suppression_incoherences_clients(
    df_ap_clients,
    controles_ap_clients,
    date_naissance="AP_BIRTH_DATE",
    date_creation="AP_DATE_CREATION",
    cle_primaire="AP_USER_ID"
)
# Idem, on travaillera avec les lignes valides
df_ap_clients = df_ap_clients_clean


# ============================================================
#  SUPPRESSION INCOHERENCES  : CONTRATS
# ============================================================

controles_contrats = [

    # Valeurs obligatoires
    (
        F.col("AB_POLICY_NUMBER").isNull() |
        (F.trim(F.col("AB_POLICY_NUMBER")) == ""),
        "Numéro de contrat manquant"
    ),

    (
        F.col("AB_CLIENT_ID").isNull() |
        (F.col("AB_CLIENT_ID") <= 0),
        "Identifiant client invalide"
    ),

    (
        F.col("AB_TYPE_ASSURANCE").isNull() |
        (~F.col("AB_TYPE_ASSURANCE").isin(
            "AUTO", "HABITATION", "SANTE",
            "VIE", "PROFESSIONNELLE", "MOTO"
        )),
        "Type assurance invalide"
    ),

    (
        F.col("AB_STATUT_CONTRAT").isNull() |
        (~F.col("AB_STATUT_CONTRAT").isin(
            "ACTIF", "RESILIE", "SUSPENDU",
            "EN_ATTENTE", "EXPIRE"
        )),
        "Statut contrat invalide"
    ),

    (
        F.col("AB_PRIME_ANNUELLE").isNull() |
        F.isnan(F.col("AB_PRIME_ANNUELLE")) |
        (F.col("AB_PRIME_ANNUELLE") < 0),
        "Prime annuelle invalide"
    ),

    (
        F.col("AB_AGENCE_ID").isNull() |
        (F.col("AB_AGENCE_ID") <= 0),
        "Identifiant agence invalide"
    )
]


# Contrôle de cohérence entre les dates
controles_contrats.append((
    F.col("__date_AB_DATE_FIN") <
    F.col("__date_AB_DATE_DEBUT"),
    "Date fin antérieure à date début"
))


# Dtection des incohérences contrats pour Ab Assurance
df_ab_contrats_clean, df_ab_contrats_Rejected = suppression_incoherences(
    df_ab_contrats,
    controles_contrats.copy(),
    colonnes_dates=["AB_DATE_DEBUT", "AB_DATE_FIN"],
    cle_primaire="AB_POLICY_NUMBER"
)
print(" les 5 premiers contrats valides de  Ab Assurance")
print(df_ab_contrats_clean.show(5))
print(" les contrats Rejected")
print(df_ab_contrats_Rejected.show(5))

# On ne travaillera  qu'avec les enregistrements valides 
df_ab_contrats= df_ab_contrats_clean

# Detection incohérences  spécifiques aux contrats AssurPlus
controles_contrats_ap = [

    # Valeurs obligatoires
    (
        F.col("AP_CONTRACT_REF").isNull() |
        (F.trim(F.col("AP_CONTRACT_REF")) == ""),
        "Numéro de contrat manquant"
    ),

    (
        F.col("AP_USER_ID").isNull() |
        (F.col("AP_USER_ID") <= 0),
        "Identifiant client invalide"
    ),

    (
        F.col("AP_PRODUCT_CODE").isNull() |
        (~F.col("AP_PRODUCT_CODE").isin(
            "Auto", "Habitation", "Santé",
            "Prévoyance"
        )),
        "Type assurance invalide"
    ),

    (
        F.col("AP_CONTRACT_STATE").isNull() |
        (~F.col("AP_CONTRACT_STATE").isin(
            "ACTIF", "RESILIE", "SUSPENDU",
            "EN_ATTENTE", "EXPIRE"
        )),
        "Statut contrat invalide"
    ),

    (
        F.col("AP_MONTHLY_PREMIUM").isNull() |
        F.isnan(F.col("AP_MONTHLY_PREMIUM")) |
        (F.col("AP_MONTHLY_PREMIUM") < 0),
        "Prime invalide"
    ),

    (
        F.col("AP_BROKER_CODE").isNull() |
        (F.col("AP_BROKER_CODE") <= 0),
        "Identifiant courtier invalide"
    ),

    # Contrôle de cohérence entre les dates
    (
        F.col("__date_AP_END_DATE") <
        F.col("__date_AP_START_DATE"),
        "Date fin antérieure à date début"
    )
]

# # Detection des contrats incohérents  pour AssurePlus
df_ap_contrats_clean, df_ap_contrats_Rejected = suppression_incoherences(
    df_ap_contrats,
    controles_contrats_ap,
    colonnes_dates=["AP_START_DATE", "AP_END_DATE"],
    cle_primaire="AP_CONTRACT_REF"
)
print(" les 5 premiers contrats valides AssurePlus")
print(df_ap_contrats_clean.show(5))
print(" les contrats Rejected")
print(df_ap_contrats_Rejected.show(5))
# On ne garde que les enregistrements valides 
df_ap_contrats= df_ap_contrats_clean


# ============================================================
# SUPPRESSION INCOHERENCES : SINISTRES
# ============================================================

controles_sinistres_ab = [

    (
        F.col("AB_CLAIM_ID").isNull() |
        (F.col("AB_CLAIM_ID") <= 0),
        "Identifiant sinistre invalide"
    ),

    (
        F.col("AB_POLICY_NUMBER").isNull() |
        (F.trim(F.col("AB_POLICY_NUMBER")) == ""),
        "Numéro de contrat manquant"
    ),

    (
        F.col("AB_MONTANT_ESTIME").isNull() |
        F.isnan(F.col("AB_MONTANT_ESTIME")) |
        (F.col("AB_MONTANT_ESTIME") < 0),
        "Montant estimé invalide"
    ),

    (
        F.col("AB_STATUT_SINISTRE").isNull() |
        (~F.col("AB_STATUT_SINISTRE").isin(
            "OUVERT", "EN_COURS", "CLOTURE",
            "REFUSE", "EN_EXPERTISE"
        )),
        "Statut sinistre invalide"
    )
]

controles_sinistres_ap = [

    (
        F.col("AP_SINISTRE_NUM").isNull() |
        (F.col("AP_SINISTRE_NUM") <= 0),
        "Identifiant sinistre invalide"
    ),

    (
        F.col("AP_CONTRACT_REF").isNull() |
        (F.trim(F.col("AP_CONTRACT_REF")) == ""),
        "Numéro de contrat manquant"
    ),

    (
        F.col("AP_ESTIMATED_AMOUNT").isNull() |
        F.isnan(F.col("AP_ESTIMATED_AMOUNT")) |
        (F.col("AP_ESTIMATED_AMOUNT") < 0),
        "Montant estimé invalide"
    ),

    (
        F.col("AP_CLAIM_STATUS").isNull() |
        (~F.col("AP_CLAIM_STATUS").isin(
            "OUVERT", "EN_COURS", "CLOTURE",
            "REFUSE", "EN_EXPERTISE"
        )),
        "Statut sinistre invalide"
    )
]



# 
df_ab_sinistres_clean, df_ab_sinistres_Rejected = suppression_incoherences(
    df_ab_sinistres,
    controles_sinistres_ab.copy(),
    colonnes_dates=["AB_DATE_SINISTRE"],
    cle_primaire="AB_CLAIM_ID",
    contrats_valides=df_ab_contrats_clean
)

print(" les sinistres valides AbAssurance")
df_ab_sinistres_clean.show(5, truncate=False)
print(" les sinistres Rejected")
print(df_ab_sinistres_Rejected.show(5,truncate = False))

  # On ne garde que les enregistrements valides 
df_ab_sinistres= df_ab_sinistres_clean



# Dtection des incohérences (sinistres) relatives à AssurePlus
df_ap_sinistres_clean, df_ap_sinistres_Rejected = suppression_incoherences(
    df_ap_sinistres,
    controles_sinistres_ap.copy(),
    colonnes_dates=["AP_INCIDENT_DATE"],
    cle_primaire="AP_SINISTRE_NUM",
    contrats_valides=df_ap_contrats_clean
)

print(" les 5 premiers  sinistres valides AssurePlus")
print(df_ap_sinistres_clean.show(5, truncate = False))
print(" les sinistres Rejected")
print(df_ap_sinistres_Rejected.show(5, truncate =  False))                                                                                                                                                                                           

# On ne garde que les enregistrements valides 
df_ap_sinistres= df_ap_sinistres_clean



# ============================================================
#SUPPRESSION INCOHERENCES : PAYMENTS
# ============================================================

controles_payments_ab = [

    (
        F.col("AB_PAYMENT_ID").isNull() |
        (F.col("AB_PAYMENT_ID") <= 0),
        "Identifiant paiement invalide"
    ),

    (
        F.col("AB_POLICY_NUMBER").isNull() |
        (F.trim(F.col("AB_POLICY_NUMBER")) == ""),
        "Numéro de contrat manquant"
    ),

    (
        F.col("AB_MONTANT").isNull() |
        F.isnan(F.col("AB_MONTANT")) |
        (F.col("AB_MONTANT") <= 0),
        "Montant paiement invalide"
    ),

    (
        F.col("AB_MODE_PAYMENT").isNull() |
        (~F.col("AB_MODE_PAYMENT").isin(
            "CARTE", "PRELEVEMENT", "VIREMENT",
            "CHEQUE", "ESPECES"
        )),
        "Mode de paiement invalide"
    )
]



controles_payments_ap = [

    (
        F.col("AP_PAYMENT_REF").isNull() |
        (F.col("AP_PAYMENT_REF") <= 0),
        "Identifiant paiement invalide"
    ),

    (
        F.col("AP_CONTRACT_REF").isNull() |
        (F.trim(F.col("AP_CONTRACT_REF")) == ""),
        "Numéro de contrat manquant"
    ),

    (
        F.col("AP_AMOUNT_PAID").isNull() |
        F.isnan(F.col("AP_AMOUNT_PAID")) |
        (F.col("AP_AMOUNT_PAID") <= 0),
        "Montant paiement invalide"
    ),

    (
        F.col("AP_PAYMENT_CHANNEL").isNull() |
        (~F.col("AP_PAYMENT_CHANNEL").isin(
            "Carte bancaire", "Prélèvement", "Virement",
            "Chèque"
        )),
        "Mode de paiement invalide"
    )
]

# Incohérences Payments AbAssurance
df_ab_payments_clean, df_ab_payments_Rejected = suppression_incoherences(
    df_ab_payments,
    controles_payments_ab.copy(),
    colonnes_dates=["AB_DATE_PAYMENT"],
    cle_primaire="AB_PAYMENT_ID",
    contrats_valides=df_ab_contrats_clean
)

print(" Les 5 premiers  payments valides AbAssurance")
print(df_ab_payments_clean.show(5, truncate = False))
print(" Les payments Rejected")
print(df_ab_payments_Rejected.show(5, truncate =  False))
# # On ne garde que les enregistrements valides 
df_ab_payments = df_ab_payments_clean



# Incohérences Payments AssurePlus
df_ap_payments_clean, df_ap_payments_Rejected = suppression_incoherences(
    df_ap_payments,
    controles_payments_ap.copy(),
    colonnes_dates=["AP_PAYMENT_DATETIME"],
    cle_primaire="AP_PAYMENT_REF",
    contrats_valides=df_ap_contrats_clean
)

print(" Les 5 premiers  payments valides AssurePlus")
print(df_ap_payments_clean.show(5, truncate = False))
print(" Les payments Rejected")
print(df_ap_payments_Rejected.show(5, truncate =  False))


# On ne garde que les enregistrements valides 
df_ap_payments = df_ap_payments_clean



#from SONAR_FONC_ENREGISTREMENT_EN_SILVER import enregistrement_silver

# Protection des données pour la couche Silver
def protection_silver(df2, source):
    salt = dbutils.secrets.get(scope="PROJET_BIG_DATA_STUDI", key="hash_salt")
    
    if source == "Oracle":
    

     df2 = df2.withColumns({"Email_Hached": F.sha2(F.concat(F.col("AB_EMAIL"), F.lit(salt)), 256),

     "Nom_Hached": F.sha2(F.concat(F.col("AB_NOM"), F.lit(salt)), 256),
     "Prenom_Hached": F.sha2(F.concat(F.col("AB_PRENOM"), F.lit(salt)), 256),
     "Adresse_Hached": F.sha2(F.concat(F.col("AB_ADRESSE"), F.lit(salt)), 256),
     "Telephone_Hached": F.sha2(F.concat(F.col("AB_TELEPHONE"), F.lit(salt)), 256),
     "Num_fiscal_Hached": F.sha2(F.concat(F.col("AB_NUM_FISCAL"), F.lit(salt)), 256),
     "Date_de_naissance_Hached": F.sha2(F.concat(F.col("AB_DATE_NAISSANCE"), F.lit(salt)), 256)}).drop("AB_NOM", "AB_PRENOM","AB_EMAIL","AB_DATE_NAISSANCE","AB_ADRESSE","AB_NUM_FISCAL","AB_TELEPHONE")
    else: 
    
     df2 = df2.withColumns({"Email_Hached": F.sha2(F.concat(F.col("AP_MAIL_ADDRESS"), F.lit(salt)), 256),
     "FULL_NAME_Hached": F.sha2(F.concat(F.col("AP_FULL_NAME"), F.lit(salt)), 256),
     "Nom_Hached": F.sha2(F.concat(F.col("AP_NOM"), F.lit(salt)), 256),
     "Prenom_Hached": F.sha2(F.concat(F.col("AP_PRENOM"), F.lit(salt)), 256),
     "Telephone_Hached": F.sha2(F.concat(F.col("AP_PHONE_NUMBER"), F.lit(salt)), 256),
     "Street_Hached": F.sha2(F.concat(F.col("AP_STREET_ADDRESS"), F.lit(salt)), 256),
     "Date_de_naissance_Hached": F.sha2(F.concat(F.col("AP_BIRTH_DATE"), F.lit(salt)), 256)}).drop("AP_FULL_NAME","AP_MAIL_ADDRESS","AP_BIRTH_DATE","AP_NOM", "AP_PRENOM","AP_PHONE_NUMBER","AP_STREET_ADDRESS")
    return df2

    


# Ajout des données pour avoir la trace des sources de provenance (AbAssurance ou AssurePlus)


from SONAR_FONC_GARDER_LA_TRACE import ajoutmétadonnées


# ORACLE
df_ab_clients =ajoutmétadonnées (df_ab_clients,"Oracle","Oracle_clients.csv","AB_CLIENT_ID")
df_ab_contrats =ajoutmétadonnées (df_ab_contrats,"Oracle","Oracle_contrats.csv","AB_POLICY_NUMBER")
df_ab_sinistres =ajoutmétadonnées (df_ab_sinistres,"Oracle","Oracle_sinistres.csv","AB_CLAIM_ID")
df_ab_payments =ajoutmétadonnées (df_ab_payments,"Oracle","Oracle_payments.csv","AB_PAYMENT_ID")


#SQL Server
df_ap_clients =ajoutmétadonnées (df_ap_clients,"SQL","SQL_clients.csv","AP_USER_ID")
df_ap_contrats =ajoutmétadonnées (df_ap_contrats,"SQL","SQL_contrats.csv","AP_CONTRACT_REF")
df_ap_sinistres =ajoutmétadonnées (df_ap_sinistres,"SQL","SQL_sinistres.csv","AP_SINISTRE_NUM")
df_ap_payments =ajoutmétadonnées (df_ap_payments,"SQL","SQL_payments.csv","AP_PAYMENT_REF")



#================================================
###   Revoir ceci 
# Étape 2 : Data Pipeline - Harmonisation & Unification (Silver Layer)

"""

Schéma client final :

MASTER_CLIENT_ID
SOURCE_SYSTEM
SOURCE_CLIENT_ID
NOM
PRENOM
NOM_COMPLET
DATE_NAISSANCE
EMAIL
TELEPHONE
ADRESSE
CODE_POSTAL
NUMERO_FISCAL
DATE_CREATION
STATUT_CLIENT
SCORE_FIDELITE
DATE_INTEGRATION
SOURCE_FILE
INGESTION_TIMESTAMP
VILLE     je ne l'ai pas rajouté


"""

# Nettoyage des données
#========================  AJOUTER            CONTRAT SINISTRE ET PAYMENTS dat cleaning 

from SONAR_FONC_NETTOYAGE_DES_DEUX_BASES import data_cleaning

df_ab_clients, df_ap_clients= data_cleaning(df_ab_clients,df_ap_clients)
 
from SONAR_FONC_ENREGISTREMENT_EN_SILVER import enregistrement_silver


#                                   ENREGISTREMENT EN SILVER        
# ATTENTION: Toutes les tables ne sont pas enregistrées dans la base de données 


# Hacher les données avant l'enregistrement
df_ab_clients_silver = protection_silver(df_ab_clients,"Oracle")
 # à faire drop 
df_ab_contrats_silver = df_ab_contrats
df_ab_sinistres_silver = df_ab_sinistres
df_ab_payments_silver= df_ab_payments

df_ap_clients_silver = protection_silver(df_ap_clients,"SQLServer")
# à faire drop 
df_ap_contrats_silver = df_ap_contrats
df_ap_sinistres_silver = df_ap_sinistres
df_ap_payments_silver= df_ap_payments


# Tables Oracle
enregistrement_silver(df_ab_clients_silver,"workspace.silver.oracle_clients")
enregistrement_silver(df_ab_contrats_silver,"workspace.silver.oracle_contrats")
enregistrement_silver(df_ab_sinistres_silver,"workspace.silver.oracle_sinistres")
enregistrement_silver(df_ab_payments_silver,"workspace.silver.oracle_payments")

# Conservation des élements Rejected pour une analyse ultérieure

enregistrement_silver(df_ab_clients_Rejected,"workspace.silver.oracle_clients_Rejected")
enregistrement_silver(df_ab_contrats_Rejected,"workspace.silver.oracle_contrats_Rejected")
enregistrement_silver(df_ab_sinistres_Rejected,"workspace.silver.oracle_sinistres_Rejected")
enregistrement_silver(df_ab_payments_Rejected,"workspace.silver.oracle_payments_Rejected")

# Tables SQL Server
enregistrement_silver(df_ap_clients_silver,"workspace.silver.sqlserver_clients")
enregistrement_silver(df_ap_contrats_silver,"workspace.silver.sqlserver_contrats")
enregistrement_silver(df_ap_payments_silver,"workspace.silver.sqlserver_payments")
enregistrement_silver(df_ap_sinistres_silver,"workspace.silver.sqlserver_sinistres")

# Conservation des élements Rejected pour une analyse ultérieure

enregistrement_silver(df_ap_clients_Rejected,"workspace.silver.sqlserver_clients_Rejected")
enregistrement_silver(df_ap_contrats_Rejected,"workspace.silver.sqlserver_contrats_Rejected")
enregistrement_silver(df_ap_payments_Rejected,"workspace.silver.sqlserver_payments_Rejected")
enregistrement_silver(df_ap_sinistres_Rejected,"workspace.silver.sqlserver_sinistres_Rejected")




In [0]:
#####   ICI NORMALEMENT COMMENCE LA COUCHE GOLD ###################################



from SONAR_FONC_HARMONISATION_ET_UNIFICATION_CLIENTS import harmonize_clients


dim_clients = harmonize_clients(df_ab_clients,df_ap_clients)
print("dim_clients les 10 premiers clients  finaux")
print(dim_clients.show(10))


# Étape 3 : Matrice de Doublons et Création de la Dimension Client

from SONAR_FONC_MATRICE_DOUBLONS import generer_matrice_doublons

# Algorithme d'identification des doublons (Matching exact sur Nom, Prénom, Date de Naissance )
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank, upper, initcap


# res_matched, res_dim, res_doublons, res_df = generer_Matrice_Doublons(dim_clients)     ceci marche 
matched_clients, dim_clients, doublons_df,result_df =generer_matrice_doublons(dim_clients)

print("Matrice des 20 premiers doublons")  #   A REVOIR
print(dim_clients.join(doublons_df, "MASTER_CLIENT_ID").show(20))

# Protection GOLD

def protection_gold(df):
    
    salt = dbutils.secrets.get(scope="PROJET_BIG_DATA_STUDI", key="hash_salt")
    
    df = df.withColumns({"Email_Hached": F.sha2(F.concat(F.col("EMAIL"), F.lit(salt)), 256),

    "Nom_Hached": F.sha2(F.concat(F.col("NOM"), F.lit(salt)), 256),
    "Prenom_Hached": F.sha2(F.concat(F.col("PRENOM"), F.lit(salt)), 256),
    "Date_de_naissance_Hached": F.sha2(F.concat(F.col("DATE_NAISSANCE"), F.lit(salt)), 256),
    "Numero_fiscal_Hached": F.sha2(F.concat(F.col("NUMERO_FISCAL"), F.lit(salt)), 256),
    "Adresse_Hached": F.sha2(F.concat(F.col("ADRESSE"), F.lit(salt)), 256),
    "Macth_key_Hached" : F.sha2(F.concat(F.col("MATCH_KEY"), F.lit(salt)), 256),
    "TELEPHONE_Hached": F.sha2(F.concat(F.col("TELEPHONE"), F.lit(salt)), 256)}).drop("NOM", "PRENOM","EMAIL","DATE_NAISSANCE", "NUMERO_FISCAL", "ADRESSE", "TELEPHONE","MATCH_KEY")

    return df

print(" ------Matrice des doublons après Hachage---------------")

dim_clients_protégés= protection_gold(dim_clients)
#dim_clients_gold = dim_clients_protege.drop("NOM","DATE_NAISSANCE","PRENOM")
dim_clients_gold = dim_clients_protégés
print("Doublons)")
dim_clients_gold.join(doublons_df, "MASTER_CLIENT_ID").show(20, truncate = False)

# Enregistrment GOLD Client


dim_clients_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.dim_clients_gold")



# Traitements  Tables de Faits Contrats et Sinistres et payments , Couche Gold

from SONAR_FONC_UNIFICATION_ET_GOLD_CONTRAT import unify_and_map_contrats

# Unification Contrats

fact_contrats_gold = unify_and_map_contrats(df_ab_contrats, df_ap_contrats, dim_clients) # ou dim_clients_gold
fact_contrats_gold.show(5,truncate=False)
# Enregistrement en Gold

fact_contrats_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.fact_contrats")   # changer le nom de la table?

print(" 5 Contrats finaux")
print(fact_contrats_gold.show(5,truncate= False))

# Unification Sinistres

from  SONAR_FONC_UNIFICATION_ET_GOLD_SINISTRE   import unify_and_map_sinistres

fact_sinistres_gold = unify_and_map_sinistres(df_ab_sinistres, df_ap_sinistres, fact_contrats_gold) 

fact_sinistres_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.fact_sinistres")   # changer le nom de la table?
print(" 5 sinistres finaux")
print(fact_sinistres_gold.show(5,truncate= False))

# Unification payments

from  SONAR_FONC_UNIFICATION_ET_GOLD_PAYMENT   import unify_and_map_payments

fact_payments_gold_mapped = unify_and_map_payments(df_ab_payments, df_ap_payments, fact_contrats_gold)

fact_payments_gold_mapped.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.fact_payments")   # changer le nom de la table?

print(" 5 Payments finaux")
print(fact_payments_gold_mapped.show(5,truncate= False))


#Dossier 3 (Suite) : SONAR_TESTs unitaires, Qualité & Dashboards

#Question 1 & 2 : SONAR_TESTs unitaires PySpark & SONAR_TEST Coverage
# Pour valider votre code (compatible avec les SONAR_TESTs unitaires PySpark) :

# voir cellule 3


# Question 3 : Identification d'un bug potentiel dans le pipeline
"""
•	Bug identifié : Clés primaires dupliquées après l'UNION simple.

•	Explication : Si AbAssurance et AssurePlus utilisent tous deux des identifiants numériques séquentiels (ID_CLIENT = 101 dans Oracle et CustomerID = 101 dans SQL Server), faire une simple jointure entraîne de fausses associations.

•	Correction : Toujours créer une clé composite (source_system + original_id) ou générer une clé surrogate unique via UUID / dense_rank() comme réalisé dans l'étape 3.

"""

"""
# Création du Dashboard directement sous Databricks
# Pour réaliser les visualisations demandées : 


#1.	Créez une nouvelle cellule SQL dans votre notebook.

#2.	Exécutez les 4 requêtes ci-dessous.

#3.	Cliquez sur l'icône "+" à côté des résultats de la cellule -> Visualization, sélectionnez le type de graphique, puis cliquez sur Add to Dashboard. 
"""
"""
#SQL
#-- 1. Répartition des clients par Ville (Graphique Camembert / Pie Chart)
SELECT ville, COUNT(DISTINCT master_client_id) as total_clients
FROM gold_dim_client
GROUP BY ville;

#-- 2. Statistiques Contrats (Graphique en Barres)
SELECT type_contrat, COUNT(contrat_id) as nb_contrats, SUM(montant_prime) as total_primes
FROM gold_fact_contrat
GROUP BY type_contrat;

#-- 3. Statistiques Sinistres & Financières (Indicateurs / Counter Widgets)
SELECT 
SUM(c.montant_prime) as Chiffre_Affaires_Total,
SUM(s.montant_indemnise) as Total_Indemnites_Versees,
ROUND((SUM(s.montant_indemnise) / SUM(c.montant_prime)) * 100, 2) as Ratio_Sinistralite_Pourcent
FROM gold_fact_contrat c
LEFT JOIN gold_fact_sinistres ON c.contrat_id = s.contrat_id;

#-- 4. Matrice des Doublons (Tableau filtré)
SELECT master_client_id, nom, prenom, COUNT(*) as nombre_de_comptes
FROM gold_dim_client
GROUP BY master_client_id, nom, prenom
HAVING COUNT(*) > 1;
"""
"""
Résumé des livrables pour votre rendu :
 

4.	Code & Dashboard : Exécutez le script PySpark complet sous Databricks Community Edition pour générer les tables Delta et publier le tableau de bord interactif.

"""

In [0]:
%sql
-- CREATE DASHBOARD: dashboard bloc 333
-- Les requêtes suivantes peuvent être ajoutées comme widgets dans le dashboard "dashboard bloc 333"

-- 1. Nombre total de clients
SELECT COUNT(DISTINCT MASTER_CLIENT_ID) AS total_clients
FROM gold_dim_clients;

-- 2. Répartition des clients par code postal
SELECT CODE_POSTAL, COUNT(DISTINCT MASTER_CLIENT_ID) AS nb_clients
FROM gold_dim_clients
GROUP BY CODE_POSTAL;

-- 3. Statistiques contrats par type
SELECT TYPE_CONTRAT, COUNT(CONTRAT_ID) AS nb_contrats, SUM(MONTANT_PRIME) AS montant_total
FROM gold_fact_contrat
GROUP BY TYPE_CONTRAT;

-- 4. Statistiques sinistres & financières
SELECT 
    SUM(c.MONTANT_PRIME) AS chiffre_affaires,
    SUM(s.MONTANT_INDEMNISE) AS indemnites_versees,
    ROUND((SUM(s.MONTANT_INDEMNISE) / SUM(c.MONTANT_PRIME)) * 100, 2) AS ratio_sinistralite_pct
FROM gold_fact_contrat c
LEFT JOIN gold_fact_sinistres 
    ON c.CONTRAT_ID = s.CONTRAT_ID;

-- 5. Matrice des doublons
SELECT MASTER_CLIENT_ID, Nom_Hached, Prenom_Hached, COUNT(*) AS nb_comptes
FROM gold_dim_clients
GROUP BY MASTER_CLIENT_ID, Nom_Hached, Prenom_Hached
HAVING COUNT(*) > 1;
--SQL
-- 1. Répartition des clients par Ville (Graphique Camembert / Pie Chart)
--69

SELECT CODE_POSTAL, COUNT(DISTINCT master_client_id) as total_clients
FROM gold_dim_clients
GROUP BY CODE_POSTAL;
-- 2. Statistiques Contrats (Graphique en Barres)

SELECT type_contrat, COUNT(contrat_id) as nb_contrats, SUM(montant_prime) as total_primes
FROM gold_fact_contrat
GROUP BY type_contrat;

-- 3. Statistiques Sinistres & Financières (Indicateurs / Counter Widgets)

SELECT
SUM(c.montant_prime) as Chiffre_Affaires_Total,
SUM(s.montant_indemnise) as Total_Indemnites_Versees,
ROUND((SUM(s.montant_indemnise) / SUM(c.montant_prime)) * 100, 2) as Ratio_Sinistralite_Pourcent
FROM gold_fact_contrat c
LEFT JOIN gold_fact_sinistres ON c.contrat_id = s.contrat_id;

-- 4. Matrice des Doublons (Tableau filtré)

SELECT master_client_id, Nom_Hached, Prenom_Hached, COUNT(*) as nombre_de_comptes
FROM gold_dim_client
GROUP BY master_client_id, Nom_Hached, Prenom_Hached
HAVING COUNT(*) > 1;

In [0]:
# Installation de coverage (requis par la cellule de tests ci-dessous)
# À relancer au début de chaque nouvelle session de notebook
%pip install coverage

In [0]:
# ============================================================
# STEP 1 : CONFIGURATION ET DÉMARRAGE IMMÉDIAT DU COVERAGE
# ============================================================
import sys
import os
from coverage import Coverage

# Définir le chemin absolu du dossier SRC
src_path = "/Workspace/Users/kacidatabricks@gmail.com/TEST_SONAR_GIT/TEST_SONAR_GIT/SRC"

test_path = "/Workspace/Users/kacidatabricks@gmail.com/TEST_SONAR_GIT/TEST_SONAR_GIT/TESTS"

if src_path not in sys.path:
    sys.path.append(src_path)

if test_path not in sys.path:
   sys.path.append(test_path)



# On initialise et on démarre TOUT DE SUITE le coverage, avant tout import !
cov = Coverage(source=[src_path])
cov.start()

# ==========================================
# STEP 2 : IMPORTS  DES MODULES 
# ==========================================

import pkgutil
import importlib

print("Importation des modules (SRC) sous surveillance de coverage...")
for module_info in pkgutil.iter_modules([src_path]):
    # Ignorer les fichiers non importables comme modules (ex. "__init__.py.py" -> nom "__init__.py" invalide)
    if not module_info.name.isidentifier():
        continue
    # Charge le fichier proprement en tant que module Python valide
    # reload : recharge le module s'il est déjà en mémoire, pour prendre en compte les modifications du fichier
    module = importlib.import_module(module_info.name)
    module = importlib.reload(module)
    
    # Injecte les fonctions dans l'espace global pour que les  tests unittest les trouvent
    module_attrs = [attr for attr in dir(module) if not attr.startswith('_')]
    for attr in module_attrs:
        globals()[attr] = getattr(module, attr)
# Pour les tests : les fichiers de TESTS sont chargés via exec() (la couverture ne porte que sur SRC).
# exec() fait partager le namespace du notebook, ce qui est indispensable ici car les fonctions
# de test s'appellent entre elles et utilisent les DataFrames créés par les fichiers d'ingestion.
import glob

print("Chargement des fichiers de tests (TESTS)...")
test_files = sorted(glob.glob(os.path.join(test_path, "SONAR_TEST_*.py")))
# Les fichiers d'ingestion en premier : ils créent df_ab_* utilisés par les autres tests
ingestion_files = [f for f in test_files if "INGESTION" in os.path.basename(f)]
other_files = [f for f in test_files if f not in ingestion_files]

for f in ingestion_files + other_files:
    try:
        exec(open(f).read())
    except Exception as e:
        print(f"ATTENTION : erreur au chargement de {os.path.basename(f)} : {e}")

# Plusieurs tests utilisent df_ab_* : on les crée via la fonction SRC
# (dans le fichier TESTS d'ingestion Oracle, la création au niveau module est dans un docstring = code mort)
df_ab_clients, df_ab_contrats, df_ab_sinistres, df_ab_payments = create_ab_dataframes(spark)
# Idem pour les DataFrames AssurePlus (df_ap_*), utilisés notamment par harmonize_clients
df_ap_clients, df_ap_contrats, df_ap_sinistres, df_ap_payments = load_ap_data(spark)

# ============================================================================
# Données synthétiques pour les tests Gold de la classe ci-dessous
# (matched_clients, dim_clients, doublons_df, fact_contrats_gold, fact_sinistres_gold)
# NB : table_name de test uniquement -> jamais les tables Gold de production.
# ============================================================================
from pyspark.sql import Row

_synthetic_clients = spark.createDataFrame([
    Row(SOURCE_SYSTEM="ORACLE", ORIGINAL_ID="1", NOM="DUPONT", PRENOM="Jean",
        DATE_NAISSANCE="1980-01-01", EMAIL="jean.dupont@example.fr", TELEPHONE="0612345678",
        ADRESSE="1 rue A", CODE_POSTAL="75001", NUMERO_FISCAL="1234567890123",
        DATE_CREATION="2020-01-01", STATUT_CLIENT="ACTIF",
        SOURCE_FILE="test.csv", INGESTION_TIMESTAMP="2026-11-16 00:00:00"),
    Row(SOURCE_SYSTEM="SQL", ORIGINAL_ID="2", NOM="DUPONT", PRENOM="Jean",
        DATE_NAISSANCE="1980-01-01", EMAIL="jean.dupont2@example.fr", TELEPHONE="0612345679",
        ADRESSE="2 rue B", CODE_POSTAL="69000", NUMERO_FISCAL="1234567890123",
        DATE_CREATION="2020-01-02", STATUT_CLIENT="ACTIF",
        SOURCE_FILE="test.csv", INGESTION_TIMESTAMP="2026-11-16 00:00:00"),
])

# Matrice de doublons : les deux clients identiques partagent le même MASTER_CLIENT_ID
matched_clients, dim_clients, doublons_df, _result_doublons = generer_matrice_doublons(_synthetic_clients)

# Contrats : un par source, chacun rattaché à son client
_synthetic_ab_contrats = spark.createDataFrame(
    [("P1", "1", "AUTO", "ACTIVE", 100.0)],
    ["AB_POLICY_NUMBER", "AB_CLIENT_ID", "AB_TYPE_ASSURANCE", "AB_STATUT_CONTRAT", "AB_PRIME_ANNUELLE"])
_synthetic_ap_contrats = spark.createDataFrame(
    [("C1", "2", "Auto", "ACTIF", 50.0)],
    ["AP_CONTRACT_REF", "AP_USER_ID", "AP_PRODUCT_CODE", "AP_CONTRACT_STATE", "AP_MONTHLY_PREMIUM"])
fact_contrats_gold = unify_and_map_contrats(_synthetic_ab_contrats, _synthetic_ap_contrats, dim_clients)

# Sinistres : un par source, écriture dans une table de TEST uniquement
_synthetic_ab_sinistres = spark.createDataFrame(
    [("1", "P1", 100.0, "OUVERT")],
    ["AB_CLAIM_ID", "AB_POLICY_NUMBER", "AB_MONTANT_ESTIME", "AB_STATUT_SINISTRE"])
_synthetic_ap_sinistres = spark.createDataFrame(
    [("2", "C1", 200.0, "CLOTURE")],
    ["AP_SINISTRE_NUM", "AP_CONTRACT_REF", "AP_ESTIMATED_AMOUNT", "AP_CLAIM_STATUS"])
fact_sinistres_gold = unify_and_map_sinistres(
    _synthetic_ab_sinistres, _synthetic_ap_sinistres, fact_contrats_gold,
    table_name="gold.fact_sinistre_test")

# Import des fonctions de tests
import unittest
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

# Charger vos fichiers de tests (Ici vous pouvez garder exec() UNIQUEMENT pour les scripts de tests,
# car nous ne cherchons pas à mesurer la couverture des fichiers de tests eux-mêmes, mais celle de SRC)
# je les commente
#exec(open("/Workspace/Users/kacidatabricks@://gmail.com").read())
#exec(open("/Workspace/Users/kacidatabricks@://gmail.com").read())

# ... (Gardez vos autres lignes d'exécution de fichiers de TESTS ici) ...



# ==========================================
# STEP 3 : EXÉCUTION DE la  CLASSE DE TEST
# ==========================================

# Fonction clean_email utilisée par test_clean_email
def clean_email(value):
    if value is None:
        return None
    return value.strip().lower()


class TestETLPipeline(unittest.TestCase):
    def test_clean_email(self):
        self.assertEqual(clean_email("  JEAN.DUPONT@Email.FR "), "jean.dupont@email.fr")
        self.assertIsNone(clean_email(None))
        
    def test_clean_last_name(self):
        df = df_ab_clients.withColumn("AB_NOM", F.lit("   martin "))
        df_clean = clean_last_name(df, "AB_NOM")
        value = df_clean.select("AB_NOM").collect()[0][0]
        self.assertEqual(value, "MARTIN")
    
    # ... (Conservez le reste de vos méthodes de tests inchangé) ...

    '''
    def test_unified_clients(self):
        self.assertIsInstance(df_clients, DataFrame)
        self.assertTrue(df_clients.count() > 0)
    '''
    

    def test_normalize_firstname(self):
        df = df_ab_clients.withColumn("AB_PRENOM", F.lit("  jean "))
        df_norm = df.withColumn("AB_PRENOM", normalize_firstname("AB_PRENOM"))
        value = df_norm.select("AB_PRENOM").collect()[0][0]
        self.assertEqual(value, "JEAN")

    def test_normalize_phone_number(self):
        df = df_ab_clients.withColumn("AB_TELEPHONE", F.lit("0612345678"))
        df_norm = normalize_phone_number(df, "AB_TELEPHONE")
        value = df_norm.select("AB_TELEPHONE").collect()[0][0]
        self.assertTrue(value.startswith("+33"))

    def test_matching_key(self):
        self.assertIn("MATCH_KEY", matched_clients.columns)
        keys = matched_clients.select("MATCH_KEY").distinct().count()
        self.assertTrue(keys > 0)

    def test_master_client_id_uniqueness(self):
        count_id = dim_clients.select("MASTER_CLIENT_ID").distinct().count()
        self.assertTrue(count_id > 0)
    
    def test_gold_fact_contrats(self):   
       #test_unify_and_map_contrats()
        self.assertIsInstance(fact_contrats_gold, DataFrame)
        self.assertTrue(fact_contrats_gold.count() > 0)

    def test_gold_fact_sinistres(self):
        self.assertIsInstance(fact_sinistres_gold, DataFrame)
        self.assertTrue(fact_sinistres_gold.count() > 0)

    def test_doublons(self):
        df = doublons_df
        self.assertTrue("MASTER_CLIENT_ID" in df.columns)
        self.assertTrue(df.count() >= 0)

# ajoutés 16102026

    def test1_create_ab_dataframes(self):
        test_create_ab_dataframes()

    #test_load_ap_data()
    def test1_load_ap_data(self):
        load_ap_data(spark)

    def test1_enregistrement_bronze(self):
        test_enregistrement_bronze()

    def test1_ajoutmétadonnées(self):
        test_ajoutmétadonnées()
# mettre ici le test de nettoyage
    def test1_enregistrement_silver(self):
       test_enregistrement_silver()
# test_harmonize_clients()


    def test1_generer_matrice_doublons(self):
      test_generer_matrice_doublons()

    def test1_save_dim_client_to_gold(self):
      test_save_dim_client_to_gold()
# test_unify_and_map_contrats()  # Test file not found - skipped

     
    def test1_unify_sinistres(self):
      test_unify_sinistres()

    def test1_unify_payments(self):
      test_unify_payments()
    

    def testt1_nettoyage_des_deux_bases(self):
        # Le test de nettoyage attend le schéma silver (NOM, PRENOM, EMAIL, ADRESSE, CODE_POSTAL, VILLE...)
        # On reproduit l'ordre du pipeline : métadonnées -> unification -> nettoyage
        # (harmonize_clients a besoin des colonnes INGESTION_TIMESTAMP et SOURCE_FILE ajoutées par ajoutmétadonnées)
        df_ab_meta = ajoutmétadonnées(df_ab_clients, "Oracle", "oracle_clients.csv", "AB_CLIENT_ID")
        df_ap_meta = ajoutmétadonnées(df_ap_clients, "SQL Server", "assureplus_clients.csv", "AP_USER_ID")
        df_unified = harmonize_clients(df_ab_meta, df_ap_meta).withColumn("VILLE", F.lit("Paris"))
        #df = test_nettoyage_des_deux_bases(df_unified)
        df=data_cleaning(df_ab_meta,df_ap_meta)

# ===== Tests ajoutés le 16/11/2026 : protection, suppression incohérences, gold contrats =====

    def test1_protection_bronze_oracle(self):
        test_protection_bronze_oracle()

    def test1_protection_bronze_sqlserver(self):
        test_protection_bronze_sqlserver()

    def test1_suppression_incoherences_clients(self):
        test_suppression_incoherences_clients()

    def test1_suppression_incoherences(self):
        test_suppression_incoherences()

    def test1_unify_and_map_contrats(self):
        test_unify_and_map_contrats()

    def test1_unify_and_map_contrats_client_absent(self):
        test_unify_and_map_contrats_client_absent()

    def test1_unify_and_map_contrats_erreur(self):
        test_unify_and_map_contrats_erreur()



# TestLoader().loadTestsFromModule va automatiquement chercher les classes Test
loader = unittest.TestLoader()
suite = loader.loadTestsFromModule(sys.modules[__name__])

# Exécution
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

# Arrêt et Rapport de couverture
cov.stop()
cov.save()

print("\n--- RAPPORT DE COUVERTURE ---")
cov.report(ignore_errors=True)



In [0]:
# 1. Activation de l'autoreload pour prendre en compte les modifs de vos fichiers .py
%load_ext autoreload
%autoreload 2

import sys
import os
import pkgutil
import importlib

# 2. Configuration du chemin vers votre dossier 'src'
current_dir = os.getcwd()
src_path = os.path.abspath(os.path.join(current_dir, "src"))

src_path = "/Workspace/Users/kacidatabricks@gmail.com/TEST_SONAR_GIT/src"

print("src_path:  ",src_path)
if src_path not in sys.path:
    sys.path.append(src_path)

SONAR_TEST_path = "/Workspace/Users/kacidatabricks@gmail.com/TEST_SONAR_GIT/SONAR_TEST"

if SONAR_TEST_path not in sys.path:
    sys.path.append(SONAR_TEST_path)

# 3. Import automatique et massif de TOUTES les SONAR_FONCtions de TOUS les fichiers
for module_info in pkgutil.iter_modules([src_path]):
    # Importer le fichier .py en tant que module
    module = importlib.import_module(module_info.name)
    
    # Récupérer tout le contenu public du fichier
    module_attrs = [attr for attr in dir(module) if not attr.startswith('_')]
    
    # Injecter les SONAR_FONCtions directement dans l'espace global du notebook
    for attr in module_attrs:
        globals()[attr] = getattr(module, attr)

print("✅ Toutes les SONAR_FONCtions ont été importées avec succès ! Vous pouvez les utiliser directement.")



import unitSONAR_TEST
from pyspark.sql import DataFrame
#from pyspark.sql import functions as F


#SONAR_FONC
exec(open("/Workspace/Users/kacidatabricks@gmail.com/TEST_SONAR_GIT/SRC/SONAR_FONC_INGESTION_DONNEES_ORACLE.py").read())

exec(open("/Workspace/Users/kacidatabricks@gmail.com/TEST_SONAR_GIT/SRC/SONAR_FONC_INGESTION_DES_DONNES_ASSUREPLUS.py").read())

exec(open("/Workspace/Users/kacidatabricks@gmail.com/TEST_SONAR_GIT/SRC/SONAR_FONC_GARDERLATRACE.py").read())

exec(open("/Workspace/Users/kacidatabricks@gmail.com/TEST_SONAR_GIT/SRC/SONAR_FONC_ENREGISTREMENT_EN_BRONZE.py").read())

exec(open("/Workspace/Users/kacidatabricks@gmail.com/TEST_SONAR_GIT/SRC/SONAR_FONC_ENREGISTREMENT_EN_SILVER.py").read())

exec(open("/Workspace/Users/kacidatabricks@gmail.com/TEST_SONAR_GIT/SRC/SONAR_FONC_NETTOYAGE_DES_DEUX_BASES.py").read())

exec(open("/Workspace/Users/kacidatabricks@gmail.com/TEST_SONAR_GIT/SRC/SONAR_FONC_HARMONISATION_ET_UNIFICATION_CLIENTS.py").read())

exec(open("/Workspace/Users/kacidatabricks@gmail.com/TEST_SONAR_GIT/SRC/SONAR_FONC_MATRICE_DOUBLONS.py").read())

exec(open("/Workspace/Users/kacidatabricks@gmail.com/TEST_SONAR_GIT/SRC/SONAR_FONC_GOLD_CLIENT.py").read())

exec(open("/Workspace/Users/kacidatabricks@gmail.com/TEST_SONAR_GIT/SRC/SONAR_FONC_UNIFICATION_ET_GOLD_CONTRAT.py").read())

exec(open("/Workspace/Users/kacidatabricks@gmail.com/TEST_SONAR_GIT/SRC/SONAR_FONC_UNIFICATION_ET_GOLD_SINISTRE.py").read())

exec(open("/Workspace/Users/kacidatabricks@gmail.com/TEST_SONAR_GIT/SRC/SONAR_FONC_UNIFICATION_ET_GOLD_PAYMENT.py").read())

# SONAR_TEST
exec(open("/Workspace/Users/kacidatabricks@gmail.com/TEST_SONAR_GIT/SONAR_TESTS/SONAR_TEST_INGESTION_DONNEES_ORACLE.py").read())

exec(open("/Workspace/Users/kacidatabricks@gmail.com/TEST_SONAR_GIT/SONAR_TESTS/SONAR_TEST_INGESTION_DES_DONNES_ASSUREPLUS.py").read())

exec(open("/Workspace/Users/kacidatabricks@gmail.com/TEST_SONAR_GIT/SONAR_TESTS/SONAR_TEST_GARDER_LA_TRACE.py").read())

exec(open("/Workspace/Users/kacidatabricks@gmail.com/TEST_SONAR_GIT/SONAR_TESTS/SONAR_TEST_ENREGISTREMENT_EN_BRONZE.py").read())

exec(open("/Workspace/Users/kacidatabricks@gmail.com/TEST_SONAR_GIT/SONAR_TESTS/SONAR_TEST_ENREGISTREMENT_EN_SILVER.py").read())

exec(open("/Workspace/Users/kacidatabricks@gmail.com/TEST_SONAR_GIT/SONAR_TESTS/SONAR_TEST_NETTOYAGE_DES_DEUX_BASES.py").read())

exec(open("/Workspace/Users/kacidatabricks@gmail.com/TEST_SONAR_GIT/SONAR_TESTS/SONAR_TEST_HARMONISATION_ET_UNIFICATION_CLIENTS.py").read())
exec(open("/Workspace/Users/kacidatabricks@gmail.com/TEST_SONAR_GIT/SONAR_TESTS/SONAR_TEST_MATRICE_DOUBLONS.py").read())

exec(open("/Workspace/Users/kacidatabricks@gmail.com/TEST_SONAR_GIT/SONAR_TESTS/SONAR_TEST_GOLD_CLIENT.py").read())

# exec(open("/Workspace/Users/kacidatabricks@gmail.com/TEST_SONAR_GIT/SONAR_TESTS/SONAR_TEST_UNIFICATION_ET_GOLD_CONTRAT.py").read())  # File not found - skipped

exec(open("/Workspace/Users/kacidatabricks@gmail.com/TEST_SONAR_GIT/SONAR_TESTS/SONAR_TEST_UNIFICATION_ET_GOLD_SINISTRE.py").read())

exec(open("/Workspace/Users/kacidatabricks@gmail.com/TEST_SONAR_GIT/SONAR_TESTS/SONAR_TEST_UNIFICATION_ET_GOLD_PAYMENT.py").read())









# Pour ouvrir tous les fichiers en py de src
#import os

#src_path = "/Workspace/Users/kacidatabricks@gmail.com/ des src et notebooks abassurance_databricks_bloc3_kit/SRC/"
# files = [f for f in os.listdir(src_path) if f.endswith(".py")]

# for fname in files:
#    exec(open(os.path.join(src_path, fname)).read())


#      IIIIIIIIIII je veux SONAR_TESTer le coverage sur le premier paquet de SONAR_TEST IIIIIIIIII

import sys
from coverage import Coverage

# 1. Initialiser et démarrer la capture de couverture
cov = Coverage(source=["/Workspace/Users/kacidatabricks@gmail.com/TEST_SONAR_GIT/SRC"])
# cov = Coverage()
cov.start()



def clean_email(value):
    if value is None:
        return None
    return value.strip().lower()

class SONAR_TESTETLPipeline(unitSONAR_TEST.SONAR_TESTCase):
    def SONAR_TEST_clean_email(self):
        self.assertEqual(clean_email("  JEAN.DUPONT@Email.FR "), "jean.dupont@email.fr")
        self.assertIsNone(clean_email(None))

    def SONAR_TEST_ajoutetadonnees(self):
        df = df_ab_clients.limit(1)
        df_meta = ajoutmétadonnées(df, "Oracle", "file.csv", "AB_CLIENT_ID")
        columns = df_meta.columns
        for colname in ["SOURCE_SYSTEM", "ORIGINAL_ID", "SOURCE_FILE", "INGESTION_TIMESTAMP"]:
            self.assertIn(colname, columns)
    '''
    def SONAR_TEST_unified_clients(self):
        self.assertIsInstance(df_clients, DataFrame)
        self.assertTrue(df_clients.count() > 0)
    '''
    def SONAR_TEST_clean_last_name(self):
        df = df_ab_clients.withColumn("AB_NOM", F.lit("   martin "))
        df_clean = clean_last_name(df, "AB_NOM")
        value = df_clean.select("AB_NOM").collect()[0][0]
        self.assertEqual(value, "MARTIN")

    def SONAR_TEST_normalize_firstname(self):
        df = df_ab_clients.withColumn("AB_PRENOM", F.lit("  jean "))
        df_norm = df.withColumn("AB_PRENOM", normalize_firstname("AB_PRENOM"))
        value = df_norm.select("AB_PRENOM").collect()[0][0]
        self.assertEqual(value, "JEAN")

    def SONAR_TEST_normalize_phone_number(self):
        df = df_ab_clients.withColumn("AB_TELEPHONE", F.lit("0612345678"))
        df_norm = normalize_phone_number(df, "AB_TELEPHONE")
        value = df_norm.select("AB_TELEPHONE").collect()[0][0]
        self.assertTrue(value.startswith("+33"))

    def SONAR_TEST_matching_key(self):
        self.assertIn("MATCH_KEY", matched_clients.columns)
        keys = matched_clients.select("MATCH_KEY").distinct().count()
        self.assertTrue(keys > 0)

    def SONAR_TEST_master_client_id_uniqueness(self):
        count_id = dim_clients.select("MASTER_CLIENT_ID").distinct().count()
        self.assertTrue(count_id > 0)

    def SONAR_TEST_gold_fact_contrat(self):
        self.assertIsInstance(fact_contrats_gold, DataFrame)
        self.assertTrue(fact_contrats_gold.count() > 0)

    def SONAR_TEST_gold_fact_sinistres(self):
        self.assertIsInstance(fact_sinistres_gold, DataFrame)
        self.assertTrue(fact_sinistres_gold.count() > 0)

    def SONAR_TEST_doublons(self):
        df = doublons_df
        self.assertTrue("MASTER_CLIENT_ID" in df.columns)
        self.assertTrue(df.count() >= 0)

# ajoutés 16102026

    def SONAR_TEST1_create_ab_dataframes(self):
        SONAR_TEST_create_ab_dataframes()

    #SONAR_TEST_load_ap_data()
    def SONAR_TEST1_enregistrement_bronze(self):
      SONAR_TEST_enregistrement_bronze()
    def SONAR_TEST1_ajoutmétadonnées(self) :
       SONAR_TEST_ajoutmétadonnées()
# mettre ici le SONAR_TEST de nettoyage
    def SONAR_TEST1_enregistrement_silver(self):
       SONAR_TEST_enregistrement_silver()
# SONAR_TEST_harmonize_clients()
    def SONAR_TEST1_generer_matrice_doublons(self):
      SONAR_TEST_generer_matrice_doublons()
    def SONAR_TEST1_save_dim_client_to_gold(self):
      SONAR_TEST_save_dim_client_to_gold()
# SONAR_TEST_SONAR_unify_and_map_contrats()  # SONAR_TEST file not found - skipped
    def SONAR_TEST1_SONAR_unify_sinistres(self):
      SONAR_TEST_SONAR_unify_sinistres()
    def SONAR_TEST1_SONAR_unify_payments(self):
      SONAR_TEST_SONAR_unify_payments()


suite = unitSONAR_TEST.SONAR_TESTLoader().loadSONAR_TESTsFromSONAR_TESTCase(SONAR_TESTETLPipeline)
runner = unitSONAR_TEST.TextSONAR_TESTRunner(verbosity=2)
runner.run(suite)
# The rest of the text is commented out as per instructions


# IIIIIIIIIIIIIIIIIIIIIIIIIIje l ai rajouté pour SONAR_TESTer le premier pasuet avec coverage
cov.stop()
cov.save()

# 4. Afficher le rapport directement dans l'output de la cellule Databricks
print("\n--- RAPPORT DE COUVERTURE DES SONAR_TESTS ---")
cov.report(ignore_errors=True)

#    FIN  IIIIIIIIIIIIIIIIIIIIIIIIIIje l ai rajouté pour SONAR_TESTer le premier pasuet avec coverage



#  IIIIIIII  Le COVERAGE  ###########################
'''
import sys
from coverage import Coverage

# 1. Initialiser et démarrer la capture de couverture
cov = Coverage()
cov.start()

# -------------------------------------------------------------
# Vos SONAR_FONCtions de SONAR_TESTs (Exemple)
# -------------------------------------------------------------
def SONAR_TEST_ma_SONAR_FONCtion_1():
    # Remplacez par votre vrai appel et assert
    assert ma_SONAR_FONCtion_a_SONAR_TESTer(2) == 4

def SONAR_TEST_ma_SONAR_FONCtion_2():
    assert ma_SONAR_FONCtion_a_SONAR_TESTer(0) == 0

# -------------------------------------------------------------
# Exécution manuelle des SONAR_FONCtions de SONAR_TEST dans la cellule
# -------------------------------------------------------------
try:
    print("Démarrage des SONAR_TESTs...")
    t = SONAR_TESTETLPipeline()
    t.SONAR_TEST_clean_email()
    t.SONAR_TEST_ajoutetadonnees()
    #t.SONAR_TEST_unified_clients()
    t.SONAR_TEST_clean_last_name()
    t.SONAR_TEST_normalize_firstname()
    t.SONAR_TEST_normalize_phone_number()
    t.SONAR_TEST_matching_key()
    t.SONAR_TEST_master_client_id_uniqueness()
    t.SONAR_TEST_gold_fact_contrat()
    t.SONAR_TEST_gold_fact_sinistre()
    t.SONAR_TEST_doublons()
# Ajout de nouveaux SONAR_TESTs##############
  
    SONAR_TEST_create_ab_dataframes()
    SONAR_TEST_load_ap_data()
    SONAR_TEST_enregistrement_bronze()
    SONAR_TEST_ajoutmétadonnées()
    # mettre ici le SONAR_TEST de nettoyage
    SONAR_TEST_enregistrement_silver()
    # SONAR_TEST_harmonize_clients()
    SONAR_TEST_generer_MATRICE_DOUBLONS()
    SONAR_TEST_save_dim_client_to_gold()
    # SONAR_TEST_SONAR_unify_and_map_contrats()  # SONAR_TEST file not found - skipped

    SONAR_TEST_SONAR_unify_sinistres()
    SONAR_TEST_SONAR_unify_payments()

    print("Tous les SONAR_TESTs ont réussi !")
except AssertionError as e:
    print(f"Échec d'un SONAR_TEST : {e}")
finally:
    # 3. Arrêter la capture même si un SONAR_TEST échoue
    cov.stop()
    cov.save()

# 4. Afficher le rapport directement dans l'output de la cellule Databricks
print("\n--- RAPPORT DE COUVERTURE DES SONAR_TESTS ---")
cov.report(ignore_errors=True)

'''


In [0]:
%sql

/*
# Création du Dashboard directement sous Databricks
# Pour réaliser les visualisations demandées : 


#1.	Créez une nouvelle cellule SQL dans votre notebook.

#2.	Exécutez les 4 requêtes ci-dessous.

#3.	Cliquez sur l'icône "+" à côté des résultats de la cellule -> Visualization, sélectionnez le type de graphique, puis cliquez sur Add to Dashboard. 

*/

/*
-- 0. Nombre total de clients dans la base
SELECT COUNT(DISTINCT MASTER_CLIENT_ID) AS total_clients
FROM workspace.default.gold_dim_client;

-- 1. Répartition des clients par code postal (Pie Chart)
SELECT CODE_POSTAL, COUNT(DISTINCT MASTER_CLIENT_ID) AS nb_clients
FROM workspace.default.gold_dim_client
GROUP BY CODE_POSTAL;

-- 2. Statistiques des contrats par type (Bar Chart)
SELECT TYPE_CONTRAT, COUNT(CONTRAT_ID) AS nb_contrats, SUM(MONTANT_PRIME) AS montant_total
FROM workspace.default.gold_fact_contrat
GROUP BY TYPE_CONTRAT;

-- 3. Statistiques sinistres et financières (KPIs)
SELECT 
    SUM(c.MONTANT_PRIME) AS chiffre_affaires,
    SUM(s.MONTANT_INDEMNISE) AS indemnites_versees,
    ROUND((SUM(s.MONTANT_INDEMNISE) / SUM(c.MONTANT_PRIME)) * 100, 2) AS ratio_sinistralite_pct
FROM workspace.default.gold_fact_contrat c
LEFT JOIN workspace.default.gold_fact_sinistres 
    ON c.CONTRAT_ID = s.CONTRAT_ID;

-- 4. Matrice des doublons (Table Filter)
SELECT MASTER_CLIENT_ID, Nom_Hached, Prenom_Hached, COUNT(*) AS nb_comptes
FROM workspace.default.gold_dim_client
GROUP BY MASTER_CLIENT_ID, Nom_Hached, Prenom_Hached
HAVING COUNT(*) > 1;

--SQL
--0. Création de l'information total_clients
SELECT COUNT(DISTINCT MASTER_CLIENT_ID) as total_clients
FROM gold_dim_client;
*/
-- 1. Répartition des clients par Ville (Graphique Camembert / Pie Chart)
SELECT CODE_POSTAL, COUNT(DISTINCT MASTER_CLIENT_ID) as total_clients
FROM gold_dim_clients
GROUP BY CODE_POSTAL;

/*
-- 2. Statistiques Contrats (Graphique en Barres)
SELECT TYPE_CONTRAT, COUNT(CONTRAT_ID) as nb_contrats, SUM(MONTANT_PRIME) as total_primes
FROM gold_fact_contrat
GROUP BY TYPE_CONTRAT;


-- 3. Statistiques Sinistres & Financières (Indicateurs / Counter Widgets)
SELECT 
SUM(c.MONTANT_PRIME) as Chiffre_Affaires_Total,
SUM(s.montant_indemnise) as Total_Indemnites_Versees,
ROUND((SUM(s.MONTANT_INDEMNISE) / SUM(c.MONTANT_PRIME)) * 100, 2) as Ratio_Sinistralite_Pourcent
FROM gold_fact_contrat c
LEFT JOIN gold_fact_sinistres ON c.CONTRAT_ID = s.CONTRAT_ID;


-- 4. Matrice des Doublons (Tableau filtré)
SELECT MASTER_CLIENT_ID, Nom_Haxhed, Prenom_Haxhed, COUNT(*) as nombre_de_comptes
FROM gold_dim_client
GROUP BY MASTER_CLIENT_ID, Nom_Hached, Prenom_Hached
HAVING COUNT(*) > 1;


-- ce que je rajoute
-- contrats 
SELECT
STATUT_CONTRAT,
COUNT(*) AS nb_contrats
FROM gold_fact_contrat
GROUP BY STATUT_CONTRAT;

-- sinistres 
SELECT
STATUT_SINISTRE,
COUNT(*) AS nb_sinistres,
SUM(MONTANT_INDEMNISE) AS montant_total
FROM gold_fact_sinistres
GROUP BY STATUT_SINISTRE;

-- finances 

SELECT
    SUM(MONTANT_PRIME) AS primes
FROM gold_fact_contrat;


SELECT
    SUM(MONTANT) AS paiements
FROM gold_fact_payment;

-- 0. Nombre total de clients
SELECT COUNT(DISTINCT MASTER_CLIENT_ID) AS total_clients
FROM gold_dim_client;

-- 1. Répartition des clients par code postal
SELECT CODE_POSTAL, COUNT(DISTINCT MASTER_CLIENT_ID) AS nb_clients
FROM gold_dim_client
GROUP BY CODE_POSTAL;

-- 2. Statistiques contrats par type
SELECT TYPE_CONTRAT, COUNT(CONTRAT_ID) AS nb_contrats, SUM(MONTANT_PRIME) AS montant_total
FROM gold_fact_contrat
GROUP BY TYPE_CONTRAT;
*/
-- 3. Statistiques sinistres & financières
SELECT 
    SUM(c.MONTANT_PRIME) AS chiffre_affaires,
    SUM(s.MONTANT_INDEMNISE) AS indemnites_versees,
    ROUND((SUM(s.MONTANT_INDEMNISE) / SUM(c.MONTANT_PRIME)) * 100, 2) AS ratio_sinistralite_pct
FROM gold_fact_contrat c
LEFT JOIN gold_fact_sinistres 
    ON c.CONTRAT_ID = s.CONTRAT_ID;
/*
-- 4. Matrice des doublons
SELECT MASTER_CLIENT_ID, Nom_Hached, Prenom_Hached, COUNT(*) AS nb_comptes
FROM gold_dim_client
GROUP BY MASTER_CLIENT_ID, Nom_Hached, Prenom_Hached
HAVING COUNT(*) > 1;

-- 5. Contrats par statut
SELECT STATUT_CONTRAT, COUNT(*) AS nb_contrats
FROM gold_fact_contrat
GROUP BY STATUT_CONTRAT;

-- 6. Sinistres par statut
SELECT STATUT_SINISTRE, COUNT(*) AS nb_sinistres, SUM(MONTANT_INDEMNISE) AS montant_total
FROM gold_fact_sinistres
GROUP BY STATUT_SINISTRE;

-- 7. Total primes
SELECT SUM(MONTANT_PRIME) AS primes
FROM gold_fact_contrat;

-- 8. Total paiements
SELECT SUM(MONTANT) AS paiements
FROM gold_fact_payment;
*/

/*
Résumé des livrables pour votre rendu :
1.	Document de cadrage/spécifications : Reprenez les sections du Dossier 1 (Sujet 1). 

2.	Schéma Data Flow : Exportez le schéma ASCII fourni dans le Dossier 3 (Sujet 1). 

3.	Agilité & Risques : Utilisez les tickets Trello et la matrice des risques rédigés dans le Sujet 2. 

4.	Code & Dashboard : Exécutez le script PySpark complet sous Databricks Community Edition pour générer les tables Delta et publier le tableau de bord interactif.

*/

